# ClimaCity Paris -- Jour 1
## Exploration batch : l'API RDD et l'API DataFrame

**Module** : Traitement de donnees massives avec Apache Spark et PySpark  
**Duree** : 1 journee (6 heures effectives)  
**Prerequis** : Python intermediaire, notions de Pandas et NumPy

---

Ce notebook couvre l'integralite du Jour 1 du projet ClimaCity Paris.  
Il se divise en deux grandes parties :

- **Partie 1 -- Matin (3 h)** : l'API bas niveau RDD, la semantique de l'evaluation paresseuse, les transformations et actions fondamentales, et une premiere lecture du Spark UI.
- **Partie 2 -- Apres-midi (3 h)** : l'API haut niveau DataFrame, le nettoyage des donnees, la jointure avec les observations meteorologiques, la persistance en memoire et l'ecriture en Parquet partitionne.

> **Convention dans ce notebook**  
> Les cellules marquees `# [EXERCICE]` contiennent une consigne. Vous devez completer le code avant de passer a la cellule suivante.  
> Les cellules marquees `# [CORRECTION]` proposent une solution possible -- ne les regardez qu'apres avoir tente.


---
## Section 0 -- Configuration de l'environnement

Toutes les constantes du projet sont declarees ici. Adaptez les chemins si necessaire,
puis executez cette cellule avant toute autre. Si erreur "Manquant" sur Parquet  créer le dossier manuellement.


In [54]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ.get("PATH", "")

import time
from pathlib import Path

# ── Chemins de donnees ──────────────────────────────────────────────────────
DATA_DIR       = Path("data")                          # racine des donnees
VELIB_RAW_DIR  = DATA_DIR / "velib" / "raw"              # CSV bruts (RDD)
VELIB_PARQ_DIR = DATA_DIR / "velib" / "parquet"          # Parquet partitionne
STATIONS_CSV   = DATA_DIR / "velib" / "stations_info.csv"
METEO_CSV      = DATA_DIR / "meteo" / "paris_montsouris_horaire.csv"
OUTPUT_DIR     = DATA_DIR / "output"                     # ecriture des resultats

# ── Parametres Spark ────────────────────────────────────────────────────────
APP_NAME       = "ClimaCity-Paris"
MASTER         = "local[10]"                              # toutes les CPU locales
SHUFFLE_PARTS  = 8                                       # adapte aux volumes du cours
SEED           = 42

# Verification des chemins
for p in [VELIB_RAW_DIR, VELIB_PARQ_DIR, STATIONS_CSV, METEO_CSV]:
    status = "OK" if p.exists() else "MANQUANT"
    print(f"[{status}] {p}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VELIB_PARQ_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] {VELIB_PARQ_DIR} (cree si absent)")
print(f"[OK] {OUTPUT_DIR} (cree si absent)")

[OK] data/velib/raw
[OK] data/velib/parquet
[OK] data/velib/stations_info.csv
[OK] data/meteo/paris_montsouris_horaire.csv
[OK] data/velib/parquet (cree si absent)
[OK] data/output (cree si absent)


---
## Section 0.1 -- Collecte des données Vélib'

Deux sources complémentaires sont nécessaires :

- **Les informations statiques des stations** (nom, coordonnées, capacité) : publiées
  en temps réel par l'API GBFS de Vélib' Métropole, sans authentification.
- **L'historique de disponibilité** (snapshots toutes les 15 min depuis 2019) :
  disponible sur le dépôt GitHub `lovasoa/historique-velib-opendata`.

Exécutez les cellules ci-dessous **une seule fois** avant le démarrage de la session
Spark. Les fichiers sont écrits dans `DATA_DIR` et relus ensuite par PySpark.

In [55]:
import requests
import json
import pandas as pd

# ── Endpoint GBFS Vélib' Métropole (sans clé, sans inscription) ──────────────
GBFS_STATION_INFO = (
    "https://velib-metropole-opendata.smovengo.cloud"
    "/opendata/Velib_Metropole/station_information.json"
)

print("Téléchargement des informations de stations...")
resp = requests.get(GBFS_STATION_INFO, timeout=15)
resp.raise_for_status()

stations = resp.json()["data"]["stations"]
print(f"  {len(stations)} stations reçues")

df_stations = pd.DataFrame(stations)[[
    "station_id", "name", "lat", "lon", "capacity", "stationCode"
]]
df_stations["station_id"] = df_stations["station_id"].astype(int)
df_stations["code_arr"] = df_stations["station_id"] // 1000

STATIONS_CSV.parent.mkdir(parents=True, exist_ok=True)
df_stations.to_csv(STATIONS_CSV, index=False, sep=";")
print(f"  Sauvegardé : {STATIONS_CSV}")
print(f"  Capacité totale du réseau : {df_stations['capacity'].sum():,} bornettes")
df_stations.head(5)

Téléchargement des informations de stations...
  1517 stations reçues
  Sauvegardé : data/velib/stations_info.csv
  Capacité totale du réseau : 49,005 bornettes


,station_id,name,lat,lon,capacity,stationCode,code_arr
0,213688169,Benjamin Godard - Victor Hugo,48.865983,2.275725,35,16107,213688
1,19179944124,Hôpital Mondor,48.798922,2.453745,28,40001,19179944
2,18795078746,Charcot - Benfleet,48.878370,2.440524,28,32304,18795078
3,36255,Toudouze - Clauzel,48.879296,2.337360,21,9020,36
4,251039991,Cassini - Denfert-Rochereau,48.837526,2.336035,25,14111,251039


In [56]:
from pathlib import Path

# Le fichier historique est fourni directement en local
# Il doit être placé manuellement dans VELIB_RAW_DIR
VELIB_CSV = VELIB_RAW_DIR / "historique_stations.csv"

if VELIB_CSV.exists():
    taille_mb = VELIB_CSV.stat().st_size / 1_048_576
    print(f"[OK] {VELIB_CSV} ({taille_mb:.0f} MB)")
else:
    print(f"[MANQUANT] {VELIB_CSV}")
    print("  → Placer historique_stations.csv dans le dossier data/velib/raw/")


[OK] data/velib/raw/historique_stations.csv (376 MB)


In [57]:
# Vérification du fichier local
fichiers = sorted(VELIB_RAW_DIR.glob("*.csv"))
taille_totale = sum(f.stat().st_size for f in fichiers) / 1_048_576
print(f"{len(fichiers)} fichier(s) présent(s) dans {VELIB_RAW_DIR}")
print(f"Taille totale sur disque : {taille_totale:.0f} MB")

for f in fichiers:
    nb_lignes = sum(1 for _ in open(f))
    print(f"  {f.name} : {nb_lignes:,} lignes")


1 fichier(s) présent(s) dans data/velib/raw
Taille totale sur disque : 376 MB
  historique_stations.csv : 5,298,821 lignes


In [51]:
# Si "paris_montsouris_horaire.csv" n'est pas disponible sur l'API de la Metropole alors il faut lancer le code ci-dessous. (et relancer la première cellule, tout doit être sur "ok")

In [52]:
import requests
import pandas as pd

METEO_CSV.parent.mkdir(parents=True, exist_ok=True)

# ── Téléchargement via Open-Meteo (gratuit, sans auth) ──────────────────────
# Coordonnées GPS de la station Paris-Montsouris
LAT, LON = 48.8214, 2.3378

params = {
    "latitude"      : LAT,
    "longitude"     : LON,
    "start_date"    : "2022-01-01",
    "end_date"      : "2023-12-31",
    "hourly"        : "temperature_2m,precipitation,windspeed_10m,relativehumidity_2m,surface_pressure",
    "timezone"      : "Europe/Paris",
    "wind_speed_unit": "ms",
}

print("Téléchargement des données météo Paris-Montsouris 2022-2023...")
resp = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=60)
resp.raise_for_status()
data = resp.json()

# Aperçu de la structure retournée
print(f"  Clés disponibles : {list(data.keys())}")
print(f"  Variables horaires : {list(data['hourly'].keys())}")
print(f"  Nombre de points   : {len(data['hourly']['time'])}")

# Conversion en DataFrame et sauvegarde
df_meteo = pd.DataFrame(data["hourly"])
df_meteo.rename(columns={
    "time"                   : "horodatage",
    "temperature_2m"          : "temperature",
    "precipitation"           : "precipitations",
    "windspeed_10m"           : "vent_ms",
    "relativehumidity_2m"     : "humidite",
    "surface_pressure"        : "pression",
}, inplace=True)

df_meteo.to_csv(METEO_CSV, index=False, sep=";")
print(f"  Sauvegardé : {METEO_CSV}")
print(f"  Lignes     : {len(df_meteo):,}")
df_meteo.head(3)

Téléchargement des données météo Paris-Montsouris 2022-2023...
  Clés disponibles : ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
  Variables horaires : ['time', 'temperature_2m', 'precipitation', 'windspeed_10m', 'relativehumidity_2m', 'surface_pressure']
  Nombre de points   : 17520
  Sauvegardé : data/meteo/paris_montsouris_horaire.csv
  Lignes     : 17,520


,horodatage,temperature,precipitations,vent_ms,humidite,pression
0,2022-01-01T00:00,9.4,0.0,3.22,94,1016.9
1,2022-01-01T01:00,9.4,0.0,3.11,94,1016.7
2,2022-01-01T02:00,10.7,0.0,2.50,95,1017.1


# **Nota bene**
- Le téléchargement complet (2 ans) représente environ 800 MB à 1,5 GB selon les périodes couvertes par les releases ; prévoir 15 à 30 minutes selon la connexion.
- L'API GitHub Releases est non authentifiée mais limitée à 60 requêtes/heure par IP. Un seul appel à REPO_API suffit pour lister tout l'historique.


---
# PARTIE 1 -- L'API RDD (matin)

## 1.1 La SparkSession et le SparkContext

### Pourquoi Spark ?

Pandas est un outil remarquable pour les donnees qui tiennent en memoire sur une seule
machine. Mais il atteint ses limites lorsque le volume depasse quelques gigaoctets, ou
lorsque le traitement doit s'executer en parallele sur plusieurs coeurs ou plusieurs noeuds.

Apache Spark repond a ce besoin en proposant un modele de calcul distribue et tolerant aux
pannes, dans lequel les donnees sont representees comme des collections immuables et
partitionnees, les **RDD** (*Resilient Distributed Datasets*).

Meme en mode local -- sur une seule machine, comme c'est le cas dans ce cours -- Spark
parallelise les traitements sur tous les coeurs disponibles et permet de travailler sur
des volumes qui depassent la memoire vive grace a la gestion automatique du debordement
sur disque.

### Le point d'entree : `SparkSession`

Depuis Spark 2.0, `SparkSession` est le point d'entree unifie. Il encapsule le
`SparkContext` (execution RDD), le `SQLContext` (SQL et DataFrame) et le
`HiveContext`. On n'en cree qu'une par application.


In [42]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel

"""
Exercice 1 :
------------
Initialiser une session Spark en assignant :
- un nombre de partitions
- une taille de mémoire
- en supprimant la trace de la console
"""
spark = (
      SparkSession.builder
      .appName(APP_NAME)
      .master(MASTER)
      .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
      .config("spark.executor.memory", "2g")
      .config("spark.ui.showConsoleProgress", "false")
      .getOrCreate()
  )
    # [Réponse]

# Le SparkContext est accessible depuis la session
sc = spark.sparkContext
sc.setLogLevel("WARN")    # reduit les logs INFO

print(f"Spark version    : {spark.version}")
print(f"Python version   : {sc.pythonVer}")
print(f"Master           : {sc.master}")
print(f"Coeurs detectes  : {sc._jsc.sc().defaultParallelism()}")
print(f"\nSpark UI disponible sur : http://localhost:4040")

Spark version    : 4.0.4
Python version   : 3.9
Master           : local[*]
Coeurs detectes  : 10

Spark UI disponible sur : http://localhost:4040


26/07/21 11:38:07 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


> **Spark UI** : ouvrez l'adresse affichee ci-dessus dans votre navigateur. Vous trouverez
> l'onglet **Jobs** (vide pour l'instant), **Stages**, **Storage** et **Environment**.
> Gardez-le ouvert en permanence pendant cette journee.

### Structure d'un cluster Spark (meme en local)

```
SparkSession (driver)
  |
  +-- SparkContext
        |
        +-- Executor 1 (Thread pool)
        |     +-- Task 1-1
        |     +-- Task 1-2
        +-- Executor 2 (Thread pool)
              +-- Task 2-1
              +-- Task 2-2
```

En mode `local[*]`, le driver et les executeurs partagent le meme processus JVM.
Chaque coeur logique correspond a un *slot* d'execution de taches.


---
## 1.2 Chargement des donnees brutes avec `sc.textFile()`

Le jeu de donnees brut se presente sous forme de fichiers CSV compresses (`.csv.gz`),
un par quinzaine de jours environ. Chaque ligne decrit l'etat d'une station a un instant
donne. Voici le format :

```
station_id;nom_station;code_arrondissement;capacite;velos_meca;velos_elec;bornettes_libres;horodatage
1001;Bir-Hakeim - Grenelle;75015;25;12;3;10;2022-01-01T00:14:52+00:00
1002;Javel - Andre Citroen;75015;30;0;0;30;2022-01-01T00:14:55+00:00
...
```

L'API `sc.textFile()` retourne un **RDD de chaines de caracteres** -- une ligne du fichier
par element. Aucune interpretation du contenu n'est effectuee a ce stade.


In [74]:
"""
Exercice 2 :
------------
Charger les fichiers de log des Velib
sc.textFile() accepte un chemin vers un fichier, un repertoire, ou un glob
Il decompresse automatiquement les .gz
"""
raw_rdd = sc.textFile(str(VELIB_CSV))
"""
Exercice 3 :
------------
Compter le nombre de lignes du RDD.
count() est une ACTION -- elle declenche le calcul
"""
print(f"Nombre de lignes : {raw_rdd.count():,}")

"""
Exercice 4 :
------------
Apercu des 5 premieres lignes -- take() est aussi une action
"""
for ligne in raw_rdd.take(5):
    print(ligne)

Nombre de lignes : 5,298,821
2020-11-26T12:59Z,35,4,5,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True
2020-11-26T12:59Z,55,23,4,André Mazet - Saint-André des Arts,"48.85376,2.33910",True
2020-11-26T12:59Z,20,0,0,Charonne - Robert et Sonia Delauney,"48.85591,2.39257",True
2020-11-26T12:59Z,21,0,1,Toudouze - Clauzel,"48.87930,2.33736",True
2020-11-26T12:59Z,30,3,1,Mairie du 12ème,"48.84086,2.38755",True


### Observations

- `sc.textFile()` est **instantane** : il ne lit pas encore les fichiers. Il cree
  seulement un *plan de calcul* (un RDD logique).
- `count()` declenche la lecture et le comptage -- c'est la premiere **action**.
- Les fichiers sont decoupes en **partitions** (une par bloc HDFS ou par fichier local).
  Le nombre de partitions determine le parallelisme maximal.

Verifions le nombre de partitions :


In [44]:
print(f"Nombre de partitions : {raw_rdd.getNumPartitions()}")

# En mode local, Spark cree typiquement une partition par coeur,
# ou une par fichier si le nombre de fichiers est superieur au nombre de coeurs.
# On peut forcer un repartitionnement :
raw_rdd_8p = raw_rdd.repartition(8)
print(f"Apres repartition(8) : {raw_rdd_8p.getNumPartitions()} partitions")
# Note : repartition() est une transformation -- elle ne s'execute pas encore.


Nombre de partitions : 12
Apres repartition(8) : 8 partitions


---
## 1.3 Transformations elementaires : `map`, `filter`

Un RDD est une collection **immuable** et **paresseuse** : les transformations ne
s'executent qu'au moment ou une action les requiert.

### Separation de l'en-tete

La premiere ligne de chaque fichier est un en-tete. Il faut l'eliminer avant de parser
les donnees.


In [48]:
"""
Exercice 5 :
------------
Recuperer l'en-tete pour connaitre les colonnes
Il existe une fonction spéciale pour cela
"""
premiere_ligne = raw_rdd.first()
print(premiere_ligne)

"""
Exercice 6 :
------------
Filtrer les lignes d'en-tete (toutes les lignes identiques a la premiere)
filter() retourne un nouveau RDD contenant uniquement les elements
pour lesquels la fonction retourne True

1) Qu'est-ce qui s'affiche à la sortie de `filter` ?
2) Comment afficher le nombre de lignes de donnees ?
"""
data_rdd = raw_rdd  
print(f"Nombre de lignes : {data_rdd.count():,}")


2020-11-26T12:59Z,35,4,5,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True
Nombre de lignes : 5,298,821


### Parsing des lignes avec `map()`

`map(f)` applique la fonction `f` a chaque element du RDD et retourne un nouveau RDD
de meme longueur contenant les resultats. C'est une transformation *un-pour-un*.


In [66]:
from datetime import datetime, timezone

COLONNES = [
    "station_id", "nom_station", "code_arr", "capacite",
    "velos_meca", "velos_elec", "bornettes_libres", "horodatage"
]

"""
Exercice 7  :
------------
"""
"""Parse une ligne CSV du fichier Velib' brut.

    Args:
        line: Chaine brute separee par des points-virgules.

    Returns:
        Dictionnaire avec les champs types, ou None si la ligne est malformee.
    """

import csv
import io

COLONNES = [ 
      "horodatage", "capacite", "velos_disponibles",
      "bornettes_libres", "nom_station", "coordonnees", "is_installed"
  ]
  
def parse_ligne(line: str):
      try:  
          champs = next(csv.reader(io.StringIO(line)))
          if len(champs) != 7:
              return None
          return {
              "horodatage"       : champs[0],
              "capacite"         : int(champs[1]),
              "velos_disponibles": int(champs[2]),
              "bornettes_libres" : int(champs[3]),
              "nom_station"      : champs[4],
              "coordonnees"      : champs[5],
              "is_installed"     : champs[6] == "True",
          }
      except Exception:
          return None

# Appliquer le parsing au jeu de données
parsed_rdd = raw_rdd.map(parse_ligne)
          
# Supprimer les lignes malformees (None)
clean_rdd = parsed_rdd.filter(lambda row: row is not None)

# Compter les lignes
print(f"Lignes totales : {raw_rdd.count():,}")
print(f"Lignes valides : {clean_rdd.count():,}")

# Afficher les deux premières
for row in clean_rdd.take(20):
    print(row)

Lignes totales : 5,298,821
Lignes valides : 5,298,821
{'horodatage': '2020-11-26T12:59Z', 'capacite': 35, 'velos_disponibles': 4, 'bornettes_libres': 5, 'nom_station': 'Benjamin Godard - Victor Hugo', 'coordonnees': '48.86598,2.27572', 'is_installed': True}
{'horodatage': '2020-11-26T12:59Z', 'capacite': 55, 'velos_disponibles': 23, 'bornettes_libres': 4, 'nom_station': 'André Mazet - Saint-André des Arts', 'coordonnees': '48.85376,2.33910', 'is_installed': True}
{'horodatage': '2020-11-26T12:59Z', 'capacite': 20, 'velos_disponibles': 0, 'bornettes_libres': 0, 'nom_station': 'Charonne - Robert et Sonia Delauney', 'coordonnees': '48.85591,2.39257', 'is_installed': True}
{'horodatage': '2020-11-26T12:59Z', 'capacite': 21, 'velos_disponibles': 0, 'bornettes_libres': 1, 'nom_station': 'Toudouze - Clauzel', 'coordonnees': '48.87930,2.33736', 'is_installed': True}
{'horodatage': '2020-11-26T12:59Z', 'capacite': 30, 'velos_disponibles': 3, 'bornettes_libres': 1, 'nom_station': 'Mairie du 12èm

> **Examen du Spark UI** : allez dans l'onglet **Jobs**. Vous devriez voir deux jobs
> termines. Cliquez sur le dernier (le `count()`). Dans la vue **Stages**, observez :
>
> - Le nombre de taches (*tasks*) -- une par partition.
> - Le temps passe dans chaque tache.
> - La colonne **Input** : la quantite de donnees lues par chaque tache.
>
> Notez que les deux `count()` que nous avons declenches ont relu les fichiers
> depuis le disque a chaque fois. C'est le comportement par defaut -- nous verrons
> comment l'eviter avec `.cache()`.


---
## 1.4 Evaluation paresseuse : transformations vs actions

C'est le concept le plus important de Spark. Toutes les **transformations** sont
**paresseuses** : elles construisent un plan de calcul (un DAG), mais ne font rien.
Seules les **actions** declenchent l'execution.

| Transformations (paresseuses) | Actions (declenchent le calcul) |
|-------------------------------|----------------------------------|
| `map()`, `filter()`, `flatMap()` | `count()`, `collect()`, `take()` |
| `groupByKey()`, `reduceByKey()` | `first()`, `top()`, `reduce()` |
| `join()`, `union()`, `distinct()` | `saveAsTextFile()`, `foreach()` |
| `repartition()`, `coalesce()` | `countByValue()`, `countByKey()` |

Illustrons ce comportement :


In [72]:
import time

# --- Construction du plan (instantane) ---
t0 = time.perf_counter()

"""
Exercice 8 :
-----------
On veut effectuer un calcul en trois étapes :
1) Relever le nombre de stations non vides (step1)
2) Calculer le nombre de vélos et le taux d'occupation (def + step2)
3) Ne conserver les lignes avec taux d'occupation de moins de 10% (step3)
"""

step1 = clean_rdd.filter(lambda r: r["capacite"] > 0)

def ajouter_taux(record):
    record = dict(record)
    record["taux_occupation"] = record["velos_disponibles"] / record["capacite"]
    return record

step2 = step1.map(ajouter_taux)

step3 = step2.filter(lambda r: r["taux_occupation"] < 0.10)

# Combien de temps a pris cette opération ?
t1 = time.perf_counter()
print(f"Construction du plan (3 transformations) : {(t1-t0)*1000:.1f} ms")

# --- Declenchement par une action ---
t2 = time.perf_counter()
"""
Exercice 9 :
------------
Compter le nombre de stations
"""
t2 = time.perf_counter()
n = step3.count()
t3 = time.perf_counter()

# Combien de temps a pris cette opération
print(f"Execution de count() (lecture + calcul) : {(t3-t2):.2f} s")
print(f"Lignes avec taux_occupation valide : {n:,}")

Construction du plan (3 transformations) : 0.4 ms
Execution de count() (lecture + calcul) : 3.18 s
Lignes avec taux_occupation valide : 1,952,492


### Le DAG (Directed Acyclic Graph)

Spark traduit votre chaine de transformations en un graphe oriente acyclique.
Ce graphe est optimise avant l'execution par le **Catalyst optimizer** (pour les
DataFrames) ou execute tel quel (pour les RDD).

```
sc.textFile()  -->  filter(header)  -->  map(parse)  -->  filter(None)
    -->  filter(capacite > 0)  -->  map(taux)  -->  filter(valide)
    -->  count()  [ACTION]
```

> **Spark UI -> Jobs -> dernier job -> DAG Visualization** : vous voyez exactement
> ce graphe. Les boites bleues sont les **stages** (separees par des shuffles).
> Une fleche grise entre deux stages indique un shuffle (transfert de donnees
> entre executeurs).


In [73]:
# On va reutiliser step3 plusieurs fois -> bonne occasion de le mettre en cache
# .cache() est equivalent a .persist(StorageLevel.MEMORY_ONLY)
step3.cache()

# Premier acces : lecture disque + mise en cache
t0 = time.perf_counter()
n1 = step3.count()
t1 = time.perf_counter()
print(f"Premier count() (lecture disque) : {t1-t0:.2f} s -- {n1:,} lignes")

# Deuxieme acces : depuis le cache en memoire
t2 = time.perf_counter()
n2 = step3.count()
t3 = time.perf_counter()
print(f"Deuxieme count() (depuis cache)  : {t3-t2:.2f} s -- {n2:,} lignes")

print(f"\nGain du cache : x{((t1-t0)/(t3-t2)):.1f}")
print("\nAllez dans Spark UI -> Storage pour voir le RDD en cache.")


Premier count() (lecture disque) : 3.73 s -- 1,952,492 lignes
Deuxieme count() (depuis cache)  : 0.21 s -- 1,952,492 lignes

Gain du cache : x17.5

Allez dans Spark UI -> Storage pour voir le RDD en cache.


---
## 1.5 Agregations avec paires cle-valeur : `reduceByKey`

L'une des operations les plus importantes de l'API RDD est `reduceByKey()`. Elle
regroupe les elements qui partagent la meme cle et combine leurs valeurs avec une
fonction associative.

### Objectif : top 10 des stations par nombre de snapshots

Nous voulons savoir quelles stations apparaissent le plus souvent dans notre historique
(indicateur indirect d'une bonne couverture de collecte).


In [75]:
"""
Exercice 10 :
------------
"""
# Etape 1 : transformer chaque enregistrement en paire (cle, valeur)
# Ici : (station_id, 1) -- on compte une occurrence par ligne
paires_rdd = clean_rdd.map(lambda r: (r["nom_station"], 1))

# Apercu
print("Exemples de paires :")
for p in paires_rdd.take(5):
    print(" ", p)


Exemples de paires :
  ('Benjamin Godard - Victor Hugo', 1)
  ('André Mazet - Saint-André des Arts', 1)
  ('Charonne - Robert et Sonia Delauney', 1)
  ('Toudouze - Clauzel', 1)
  ('Mairie du 12ème', 1)


In [77]:
# Etape 2 : sommer les occurrences par station
# reduceByKey(f) applique f a toutes les valeurs qui ont la meme cle,
# en garantissant que f est appliquee localement avant le shuffle (combinaison locale)
comptage_rdd = paires_rdd.reduceByKey(lambda a, b: a + b)

# Etape 3 : trier par nombre decroissant et prendre le top 10
top10_rdd = comptage_rdd.sortBy(lambda r: r[1], ascending=False)

# Étape 4 : Prendre les dix preùières stations
top10 = top10_rdd.take(10)

print("Top 10 des stations par nombre de snapshots :")
print(f"{'Station ID':<12} {'Snapshots':>12}")
print("-" * 26)
for station_id, count in top10:
    print(f"{station_id:<12} {count:>12,}")


Top 10 des stations par nombre de snapshots :
Station ID      Snapshots
--------------------------
Université          7,586
Place de l'Eglise        7,586
Château - République        7,586
Place de l'Europe        7,586
Invalides - Duroc        3,793
Chazelles - Courcelles         3,793
Le Brix et Mesmin - Jourdan        3,793
Bir Hakeim          3,793
Boétie - Ponthieu        3,793
Caire - Dussoubs        3,793


### `reduceByKey` vs `groupByKey` : une difference critique

`groupByKey()` regroupe *toutes* les valeurs en memoire avant de les reduire.
Sur de gros volumes, cela peut provoquer des `OutOfMemoryError`.
`reduceByKey()` effectue une **combinaison locale** avant le shuffle, ce qui reduit
drastiquement le volume de donnees transferees.

```
groupByKey()   : SHUFFLE toutes les valeurs -> reduire
reduceByKey()  : reduire localement -> SHUFFLE resultats partiels -> reduire
```

Regle pratique : **ne jamais utiliser `groupByKey()` quand `reduceByKey()` suffit**.


In [82]:
# Illustration de la difference de performance
# (Sur les petits volumes du cours, l'ecart peut etre faible -- il explose a l'echelle)

"""
Exercice 12 :
------------
Comparaison du temps d'execution de groupByKey() et reduceByKey() pour compter les occurrences par clef
"""
import time
  
  # groupByKey
t0 = time.perf_counter()
paires_rdd.groupByKey().mapValues(sum).count()
t_group = time.perf_counter() - t0
  
  # reduceByKey
t0 = time.perf_counter()
paires_rdd.reduceByKey(lambda a, b: a + b).count()
t_reduce = time.perf_counter() - t0
  
print(f"groupByKey + mapValues(sum) : {t_group:.2f} s")
print(f"reduceByKey                 : {t_reduce:.2f} s")
print(f"Rapport                     : x{t_group/t_reduce:.1f}")


groupByKey + mapValues(sum) : 3.24 s
reduceByKey                 : 3.10 s
Rapport                     : x1.0


---
## 1.6 `flatMap` et `join` entre deux RDD

### `flatMap` : une entree, zero ou plusieurs sorties

Contrairement a `map` (toujours une sortie par entree), `flatMap` peut retourner
un nombre quelconque d'elements pour chaque element d'entree. C'est utile pour
"eclater" des structures imbriquees.


In [87]:
# Exemple pedagogique : a partir de chaque snapshot, on veut produire
# autant de paires (heure_de_la_journee, taux_occupation) que necessaire
# pour alimenter une distribution horaire.

def extraire_heure_et_taux(record):
      try:
          heure = int(record["horodatage"][11:13])
          taux = record["taux_occupation"]
          return [(heure, taux)]
      except Exception:
          return []   


# Question : Quelle est la forme du résultat `heure_taux_rdd`
heure_taux_rdd = step3.flatMap(extraire_heure_et_taux)

  # --> flatMap produit une paire (heure, taux) par snapshot

"""
Exercice 13 :
-------------
# Calculer la moyenne du taux d'occupation par heure de la journee
# Astuce : on somme (taux, 1) puis on divise
"""
  # Sommer (taux, 1) par heure
somme_count_rdd = heure_taux_rdd.map(lambda x: (x[0], (x[1], 1)))
agregat_rdd     = somme_count_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
moyenne_rdd     = agregat_rdd.map(lambda x: (x[0], x[1][0] / x[1][1]))
profil_horaire  = sorted(moyenne_rdd.collect())

print("Taux d'occupation moyen par heure (stations < 10% de vélos)")
print(f"{'Heure':<8} {'Taux moyen':>10}")
print("-" * 20)
for heure, taux in profil_horaire:
    barre = "#" * int(taux * 30)
print(f"  {heure:02d}h    {taux:.4f}  {barre}")


Taux d'occupation moyen par heure (stations < 10% de vélos)
Heure    Taux moyen
--------------------
  23h    0.0369  #


### `join` entre deux RDD

On peut joindre deux RDD de paires `(cle, valeur)` comme on joinderait deux tables.


In [93]:
"""
Exercice 14 :
-------------
"""
# RDD des stations avec leur nom : (station_id, nom_station)
noms_rdd = clean_rdd.map(lambda r: (r["nom_station"], r["nom_station"]))


# RDD du total de velos par station : (station_id, total_velos_disponibles)
velos_rdd = clean_rdd.map(lambda r: (r["nom_station"],r["velos_disponibles"]))
velos_rdd = velos_rdd.reduceByKey(lambda a, b: a + b)

# Join interne


# Top 5 des stations par volume de velos disponibles (cumule)
top5 = velos_rdd.sortBy(lambda x: x[1], ascending=False).take(5)

print("Top 5 des stations par volume cumulé de vélos disponibles :")
for nom, total in top5:
    print(f"  {nom:<45} {total:>10,}")


Top 5 des stations par volume cumulé de vélos disponibles :
  Grenelle - Dr Finlay                             210,904
  Emeriau - Beaugrenelle                           208,373
  Route de Sèvres - Porte de Bagatelle             191,298
  Musée d'Orsay                                    176,383
  Parc André Citroën                               168,629


---
## 1.7 Comparaison Pandas vs Spark

Reprenons le calcul "top 10 des stations par nombre de snapshots" en Pandas et
comparons les temps et la demarche.


In [97]:
import pandas as pd
import time

# --- Version Pandas ---
t0 = time.perf_counter()

df_pandas = pd.read_csv(VELIB_CSV, header=None, names=COLONNES)
top10_pandas = df_pandas["nom_station"].value_counts().head(10)

t_pandas = time.perf_counter() - t0

# --- Version Spark (déjà calculée) ---
t0 = time.perf_counter()
top10_spark = top10_rdd.take(10)
t_spark = time.perf_counter() - t0

print(f"Pandas : {t_pandas:.2f} s  --  {len(df_pandas):,} lignes chargées")
print(f"Spark  : {t_spark:.2f} s  --  depuis le cache RDD")
print()
print("Comparaison des résultats (top 5) :")
print(f"{'Station':<45} {'Pandas':>8}  {'Spark':>8}")
print("-" * 65)
for i in range(5):
    p_nom, p_cnt = top10_pandas.index[i], top10_pandas.iloc[i]
    s_nom, s_cnt = top10_spark[i]
    print(f"{p_nom:<45} {p_cnt:>8,}  {s_cnt:>8,}")


Pandas : 2.40 s  --  5,298,821 lignes chargées
Spark  : 0.10 s  --  depuis le cache RDD

Comparaison des résultats (top 5) :
Station                                         Pandas     Spark
-----------------------------------------------------------------
Place de l'Europe                                7,586     7,586
Place de l'Eglise                                7,586     7,586
Université                                       7,586     7,586
Château - République                             7,586     7,586
Place Edmond Michelet                            3,793     3,793


In [98]:
# --- Analyse de la memoire ---
mem_pandas_mb = df_pandas.memory_usage(deep=True).sum() / 1_048_576
print(f"\nMemoire occupee par le DataFrame Pandas  : {mem_pandas_mb:.1f} MB")
print(f"Memoire disponible pour le reste de l'OS : limitee")
print()
print("Conclusion :")
print("  - Sur ce volume, Pandas est souvent plus rapide (pas d'overhead JVM/reseau)")
print("  - Spark devient avantageux quand :")
print("    * Le volume depasse la RAM disponible")
print("    * Le calcul peut etre parallelise sur un cluster")
print("    * On combine batch + streaming + ML dans le meme pipeline")
print("    * On travaille en equipe sur un cluster partage")


Memoire occupee par le DataFrame Pandas  : 1318.3 MB
Memoire disponible pour le reste de l'OS : limitee

Conclusion :
  - Sur ce volume, Pandas est souvent plus rapide (pas d'overhead JVM/reseau)
  - Spark devient avantageux quand :
    * Le volume depasse la RAM disponible
    * Le calcul peut etre parallelise sur un cluster
    * On combine batch + streaming + ML dans le meme pipeline
    * On travaille en equipe sur un cluster partage
